# Tree Models Training & Evaluation

### 1. Environment Setup

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path to enable absolute imports
root_path: Path = Path.cwd().parent.parent.parent
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

root_path

### 2. Import Required Libraries & Setup Configs

In [ ]:
import platform
import warnings
from importlib import metadata
from time import time

import dagshub
import lightgbm as lgb
import matplotlib.pyplot as plt
import mlflow
import pandas as pd
from catboost import CatBoostClassifier
from mlflow.data import pandas_dataset
from prefect.settings import PREFECT_LOGGING_TO_API_WHEN_MISSING_FLOW, temporary_settings
from sklearn.ensemble import (
    AdaBoostClassifier,
    RandomForestClassifier,
)
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

from src.configs.settings import Settings, get_settings
from src.pipeline.ingestion import load_dataset

# Ignore UserWarnings
warnings.filterwarnings("ignore", category=UserWarning)

# Load configuration settings
settings: Settings = get_settings()

# Package requirements
pip_requirements: list[str] = [
    "catboost==1.2.10",
    "dagshub==0.7.0",
    "dvc-s3==3.3.0",
    "evidently==0.7.21",
    "fastapi==0.136.0",
    "ipykernel==7.2.0",
    "ipywidgets==8.1.8",
    "lightgbm==4.6.0",
    "mlflow==3.11.1",
    "pandas==2.3.3",
    "pandera==0.31.1",
    "pip==26.0.1",
    "prefect==3.6.27",
    "scikit-learn==1.8.0",
    "skops==0.14.0",
    "streamlit==1.56.0",
    "xgboost==3.2.0",
]

# Trusted skops types
skops_trusted_types: list[str] = [
    "collections.OrderedDict",
    # CatBoost
    "catboost.core.CatBoost",
    "catboost.core.CatBoostClassifier",
    "catboost.core.CatBoostRegressor",
    # LightGBM
    "lightgbm.basic.Booster",
    "lightgbm.sklearn.LGBMClassifier",
    "lightgbm.sklearn.LGBMRegressor",
    # XGBoost
    "xgboost.core.Booster",
    "xgboost.sklearn.XGBClassifier",
    "xgboost.sklearn.XGBRegressor",
]

# Define data file paths
dataset_filepath: Path = settings.EXPERIMENTS_DATA_DIR / settings.TREE_FILENAME

# DagsHub repository configuration
repo_owner: str | None = settings.DAGSHUB_REPO_OWNER
repo_name: str | None = settings.DAGSHUB_REPO_NAME

# MLflow experiment and run naming
exp_name: str = "Comparison: Tree Models"
run_name: str = "tree_models_run"

# Number of top features to display in importance plots
top_features: int = 10

# Fixed random state for reproducibility across runs
random_state: int = 42

### 3. Setup DagsHub & MLFlow

In [ ]:
# Initialize DagsHub integration with MLflow tracking
dagshub.init(repo_name, repo_owner, mlflow=True)

# Set the active MLflow experiment
mlflow.set_experiment(exp_name)
print(f"DagsHub + MLflow initialised (experiment='{exp_name}')")

### 4. Load the Preprocessed Dataset

In [ ]:
target_col: str = "Irrigation_Need"

try:
    with temporary_settings(updates={PREFECT_LOGGING_TO_API_WHEN_MISSING_FLOW: "ignore"}):
        df: pd.DataFrame = load_dataset(dataset_filepath)
        X = df.drop(columns=[target_col])
        y = df[target_col]
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

except Exception as e:
    sys.stderr.write(f"Ingestion failed: {e}\n")
    raise

### 5. Train Models & Log to MLflow

In [ ]:
# Dictionary of baseline models to evaluate
models: dict = {
    "Decision Tree": DecisionTreeClassifier(random_state=random_state),
    "AdaBoost": AdaBoostClassifier(random_state=random_state),
    "Catboost": CatBoostClassifier(verbose=0, random_state=random_state),
    "LightGBM": lgb.LGBMClassifier(random_state=random_state),
    "XGBoost": XGBClassifier(eval_metric="mlogloss", random_state=random_state),
    "Random Forest": RandomForestClassifier(random_state=random_state),
}

In [ ]:
# Start the parent MLflow run for the entire baseline experiment
with mlflow.start_run(run_name=run_name) as parent_run:
    # Enable System Metrics
    mlflow.enable_system_metrics_logging()

    # Add Run Description
    mlflow.set_tag(
        "mlflow.note.content",
        (
            "Baseline experiment comparing Decision Tree, Random Forest, "
            "AdaBoost, Catboost, LightGBM, and XGBoost for irrigation prediction. "
            "Tracks system metrics, logs input datasets, uses MLflow Tracing for inference steps, "
            "and automatically registers the best-performing model."
        ),
    )

    # Cast integer columns to float64 to fix dataset-schema warnings
    train_df_float = pd.concat([X_train, y_train], axis=1)
    test_df_float = pd.concat([X_test, y_test], axis=1)

    int_cols = train_df_float.select_dtypes(include="int8").columns

    train_df_float[int_cols] = train_df_float[int_cols].astype("float64")
    test_df_float[int_cols] = test_df_float[int_cols].astype("float64")

    # Log Datasets with Context
    train_dataset = pandas_dataset.from_pandas(train_df_float, source=train_df_float, name="irrigation_train")
    mlflow.log_input(train_dataset, context="training")

    test_dataset = pandas_dataset.from_pandas(test_df_float, source=test_df_float, name="irrigation_test")
    mlflow.log_input(test_dataset, context="validation")

    # Log experiment-level metadata & tags
    mlflow.set_tags({"project": "baseline_comparison", "dataset_version": "v1.0", "engineer": "Vipin Kumar"})

    # Log dataset info & environment reproducibility parameters
    mlflow.log_params(
        {
            "train_rows": len(X_train),
            "test_rows": len(X_test),
            "n_features": X_train.shape[1],
            "feature_names": ", ".join(X_train.columns.tolist()),
            "target_distribution_train": y_train.value_counts().to_dict(),
            "target_distribution_test": y_test.value_counts().to_dict(),
            "random_state": random_state,
            "python_version": platform.python_version(),
            "sklearn_version": metadata.version("scikit-learn"),
            "mlflow_version": metadata.version("mlflow"),
            "os_platform": platform.platform(),
        }
    )

    results: dict = {}
    run_start_time: int | float = time()

    # Iterate through each model for training & evaluation
    for name, model in models.items():
        print(f"\n🔹 Training {name}...")
        model_start_time: int | float = time()

        with mlflow.start_run(run_name=name, nested=True) as child_run:
            # Log model hyperparameters automatically
            mlflow.log_params(model.get_params())

            # Fit model
            model.fit(X_train, y_train)

            # MLflow Tracing for Inference & Evaluation
            with mlflow.start_span(name=f"{name}_inference_evaluation"):
                y_pred = model.predict(X_test)

                # Define Model Signature
                X_test_float = X_test.astype("float64")
                signature = mlflow.models.infer_signature(X_test_float, y_pred)

                # Calculate core classification metrics
                acc = accuracy_score(y_test, y_pred)
                prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
                rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
                f1_w = f1_score(y_test, y_pred, average="weighted", zero_division=0)
                f1_m = f1_score(y_test, y_pred, average="macro", zero_division=0)

                # Calculate probabilistic metrics (ROC-AUC & Log Loss) if supported
                try:
                    y_proba = model.predict_proba(X_test)
                    roc_auc = roc_auc_score(y_test, y_proba, multi_class="ovr", average="weighted")
                    logloss = log_loss(y_test, y_proba)
                except (AttributeError, NotImplementedError):
                    roc_auc, logloss = None, None

            training_time: int | float = time() - model_start_time

            # Log all metrics to MLflow
            mlflow.log_metrics(
                {
                    "accuracy": acc,
                    "precision_weighted": prec,
                    "recall_weighted": rec,
                    "f1_weighted": f1_w,
                    "f1_macro": f1_m,
                    "training_time_sec": training_time,
                }
            )

            if roc_auc is not None:
                mlflow.log_metric("roc_auc_ovr", roc_auc)
            if logloss is not None:
                mlflow.log_metric("log_loss", logloss)

            # Feature Importance via Permutation Importance
            perm_importance = permutation_importance(
                model, X_test, y_test, n_repeats=5, random_state=random_state, n_jobs=-1
            )
            feat_imp_series = pd.Series(perm_importance.importances_mean, index=X_train.columns).sort_values(
                ascending=True
            )

            # Plot & log feature importance
            fig_imp, ax_imp = plt.subplots(figsize=(8, 6))
            feat_imp_series.tail(top_features).plot(kind="barh", ax=ax_imp, color="skyblue")
            ax_imp.set_title(f"Top {top_features} Feature Importance: {name}")
            plt.tight_layout()
            mlflow.log_figure(fig_imp, f"plots/{name}_feature_importance.png")
            plt.close(fig_imp)

            # Log Model with Sklearn Flavor
            model_info = mlflow.sklearn.log_model(
                model,
                name="model",
                signature=signature,
                serialization_format="skops",
                skops_trusted_types=skops_trusted_types,
                pip_requirements=pip_requirements,
            )

            # Log classification report as text
            mlflow.log_text(classification_report(y_test, y_pred), f"reports/{name}_report.txt")

            # Log confusion matrix plot
            fig_cm, ax_cm = plt.subplots(figsize=(6, 5))
            ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax_cm, cmap="Blues")
            ax_cm.set_title(f"Confusion Matrix: {name}")
            plt.tight_layout()
            mlflow.log_figure(fig_cm, f"plots/{name}_confusion_matrix.png")
            plt.close(fig_cm)

            print(f"{name} | Acc: {acc:.4f} | F1-W: {f1_w:.4f} | Time: {training_time:.2f}s")
            results[name] = {"accuracy": acc, "model_uri": model_info.model_uri}

    # Generate Comparison Plot of all models
    if results:
        fig_comp, ax_comp = plt.subplots(figsize=(10, 6))
        pd.Series({k: v["accuracy"] for k, v in results.items()}).sort_values().plot(
            kind="barh", ax=ax_comp, color="teal"
        )
        ax_comp.set_title("Baseline Model Comparison (Accuracy)")
        ax_comp.set_xlabel("Accuracy Score")
        plt.tight_layout()
        mlflow.log_figure(fig_comp, "plots/overall_model_comparison.png")
        plt.close(fig_comp)

        # Log total experiment duration & identify best model
        total_time: int | float = time() - run_start_time
        mlflow.log_metric("total_experiment_time_sec", total_time)

        best_model_name = max(results, key=lambda k: results[k]["accuracy"])
        best_model_uri = results[best_model_name]["model_uri"]

        print(f"\nRegistering best model: {best_model_name} (uri={best_model_uri})...")
        try:
            registered_model = mlflow.register_model(model_uri=best_model_uri, name="best_tree_model")
            print(f"Successfully registered as '{registered_model.name}' version {registered_model.version}")
        except Exception as e:
            print(f"Model registration skipped/failed: {e}")

        print(
            f"\nBaseline Complete. Best Model: {best_model_name} ({results[best_model_name]['accuracy']:.4f})"
        )